# 03 — Campaign Profitability & Customer Prioritisation

**Catalog Campaign Profitability & Customer Targeting Analysis**

## Purpose

Convert model predictions into the numbers management actually decides on:
expected revenue, gross profit, campaign cost, net profit and ROI — then rank the
250 prospects by economic value.

## The financial model

```
Expected Revenue (customer)  = Predicted Sale Amount x P(response)
Gross Profit     (customer)  = Expected Revenue x Gross Margin
Catalog Cost     (customer)  = Cost per Catalog
Net Profit       (customer)  = Gross Profit - Catalog Cost
Campaign Cost                = Number of Catalogs x Cost per Catalog
Campaign ROI                 = Net Profit / Campaign Cost
```

### Why revenue is weighted by response probability

A prospect with a $1,000 predicted sale who responds 10% of the time is worth
$100 to the campaign, not $1,000. Weighting by probability is what turns a
prediction about *purchase size* into a forecast of *campaign revenue*
(BRule-03).

### Why gross profit rather than revenue

Revenue does not fund the campaign. Only the margin on it does. Using revenue
would overstate campaign value by a factor of two at a 50% margin (BRule-02).

## Business assumptions (carried unchanged from the source project)

| Assumption | Value | Register ID |
|---|---|---|
| Cost per catalog | $6.50 | A-01 |
| Gross margin | 50% | A-02 |
| Response probability | Supplied `Score_Yes` field | A-05 |

**Requirements addressed:** BR-001 to BR-004, BR-006, BR-008, BR-010.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from src import data_cleaning as dc
from src import modeling as md
from src import profitability as pf

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

customers = dc.clean_customers(dc.load_customers())
mailing = dc.clean_mailing_list(dc.load_mailing_list())

model = md.fit_linear_model(customers)
scored = md.score_mailing_list(model, mailing)

print(f"Scored {len(scored)} campaign prospects")
print(f"Predicted sale amount: mean ${scored['Predicted_Sale_Amount'].mean():,.2f}, "
      f"range ${scored['Predicted_Sale_Amount'].min():,.2f} to ${scored['Predicted_Sale_Amount'].max():,.2f}")

## 1. Response probability

`Score_Yes` is supplied with the mailing list. It is **not** produced by this
project, and the model behind it is not available for inspection.

This matters enough to state plainly: it is treated as a given input
(assumption **A-05**), used as supplied, capped at 1.0 (BRule-04), and
stress-tested across a 60%-120% band in notebook 04 rather than taken on trust.
No alternative probability methodology is invented, because nothing in the data
would support one.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(scored["Score_Yes"], bins=30, color="#2f855a", edgecolor="white")
ax.axvline(scored["Score_Yes"].mean(), color="#c53030", linestyle="--",
           label=f"Mean {scored['Score_Yes'].mean():.1%}")
ax.set_title("Supplied response probability across the 250 prospects")
ax.set_xlabel("P(response)"); ax.set_ylabel("Prospects"); ax.legend()
plt.tight_layout(); plt.show()

print(f"Mean:   {scored['Score_Yes'].mean():.1%}")
print(f"Median: {scored['Score_Yes'].median():.1%}")
print(f"Range:  {scored['Score_Yes'].min():.1%} to {scored['Score_Yes'].max():.1%}")

## 2. Per-customer economics

In [ ]:
assumptions = pf.CampaignAssumptions(cost_per_catalog=6.50, gross_margin=0.50)
economics = pf.build_customer_economics(scored, assumptions)

display_cols = ["Customer_ID", "Customer_Segment", "Avg_Num_Products_Purchased",
                "Predicted_Sale_Amount", "Response_Probability", "Expected_Revenue",
                "Gross_Profit", "Catalog_Cost", "Expected_Net_Profit", "Customer_ROI"]
economics[display_cols].head(10)

### Worked example — reading a single row

Take a prospect with a predicted sale amount of $800 and a response probability
of 0.30:

```
Expected Revenue = $800.00 x 0.30   = $240.00
Gross Profit     = $240.00 x 0.50   = $120.00
Catalog Cost                        =   $6.50
Net Profit       = $120.00 - $6.50  = $113.50
Customer ROI     = $113.50 / $6.50  =   17.5x
```

Every figure in the campaign summary is a sum of rows computed exactly this way.

## 3. Campaign-level results

In [ ]:
summary = pf.campaign_summary(economics, assumptions)

headline = pd.DataFrame([
    {"Measure": "Customers mailed", "Value": f"{summary['customers']:,}"},
    {"Measure": "Sum of predicted sale amounts", "Value": f"${summary['predicted_sales_total']:,.2f}"},
    {"Measure": "Average response probability", "Value": f"{summary['avg_response_probability']:.1%}"},
    {"Measure": "Expected revenue", "Value": f"${summary['expected_revenue']:,.2f}"},
    {"Measure": f"Gross profit (at {summary['gross_margin']:.0%} margin)", "Value": f"${summary['gross_profit']:,.2f}"},
    {"Measure": "Campaign cost", "Value": f"${summary['campaign_cost']:,.2f}"},
    {"Measure": "EXPECTED NET PROFIT", "Value": f"${summary['net_profit']:,.2f}"},
    {"Measure": "Campaign ROI", "Value": f"{summary['roi']:.2f}x ({summary['roi']:.0%})"},
    {"Measure": "Revenue per customer mailed", "Value": f"${summary['revenue_per_customer']:,.2f}"},
    {"Measure": "Net profit per customer mailed", "Value": f"${summary['net_profit_per_customer']:,.2f}"},
])
headline

## 4. Validation against the source project

The original *Predicting Catalog Demand* analysis reported three figures. Because
the underlying datasets were supplied with this project, those figures can be
**independently recomputed** rather than quoted. BRule-13 requires that check
before any inherited number is published.

In [ ]:
source_reported = {
    "Predicted average sales (sum)": 138_292.13,
    "Predicted revenue": 47_224.87,
    "Predicted profit": 21_987.44,
}
recomputed = {
    "Predicted average sales (sum)": summary["predicted_sales_total"],
    "Predicted revenue": summary["expected_revenue"],
    "Predicted profit": summary["net_profit"],
}

validation = pd.DataFrame([
    {"Figure": k,
     "Source project": source_reported[k],
     "Recomputed here": recomputed[k],
     "Difference": recomputed[k] - source_reported[k],
     "Match": "YES" if abs(recomputed[k] - source_reported[k]) < 0.01 else "NO"}
    for k in source_reported
])
validation

### Validation outcome

All three figures reproduce to within rounding. The source project's results are
**independently verified**, not merely repeated, and can be reported as this
project's own findings.

The profit calculation also reconciles exactly:

```
$47,224.87 x 0.50  =  $23,612.44   gross profit
$23,612.44 - $1,625.00 =  $21,987.44   net profit   ✓
```

## 5. Where does the value come from?

Two questions management will ask: which segments carry the campaign, and is the
profit concentrated or spread evenly?

In [ ]:
segment = (economics.groupby("Customer_Segment")
           .agg(Customers=("Customer_ID", "count"),
                Avg_Predicted_Sale=("Predicted_Sale_Amount", "mean"),
                Avg_Response_Prob=("Response_Probability", "mean"),
                Expected_Revenue=("Expected_Revenue", "sum"),
                Gross_Profit=("Gross_Profit", "sum"),
                Expected_Net_Profit=("Expected_Net_Profit", "sum"))
           .reset_index())
segment["Campaign_Cost"] = segment["Customers"] * assumptions.cost_per_catalog
segment["ROI"] = segment["Expected_Net_Profit"] / segment["Campaign_Cost"]
segment = segment.sort_values("Expected_Net_Profit", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(segment["Customer_Segment"], segment["Expected_Net_Profit"], color="#2b6cb0")
axes[0].set_title("Expected net profit by segment")
axes[0].set_xlabel("Expected net profit ($)")
axes[1].barh(segment["Customer_Segment"], segment["ROI"], color="#2f855a")
axes[1].set_title("ROI by segment (return per dollar of catalog spend)")
axes[1].set_xlabel("ROI (x)")
plt.tight_layout(); plt.show()

segment

### Business interpretation

**Profit volume and profit efficiency are different things, and they point in
different directions.**

| Segment | Prospects | Net profit | ROI |
|---|---|---|---|
| Credit Card Only | 82 | $9,005 | 16.9x |
| Loyalty Club Only | 122 | $7,850 | 9.9x |
| Loyalty Club and Credit Card | 26 | $4,603 | 27.2x |
| Store Mailing List | 20 | $530 | 4.1x |

- **Credit Card Only** contributes the most total profit ($9,005) purely because
  it is a large slice of the list.
- **Loyalty Club and Credit Card** is by far the most *efficient*: 26 prospects,
  10% of the list, generating 21% of the profit at 27.2x return. Each catalog
  sent to this segment returns nearly seven times what a Store-Mailing-List
  catalog returns.
- **Store Mailing List** is still profitable at 4.1x, so there is no case for
  excluding it from *this* campaign — but it is the obvious place to cut first if
  budget is constrained, and the weakest candidate for list-building investment.

**Strategic implication.** The lever with the highest return is not better
targeting within this list. It is growing the dual-relationship segment.

In [ ]:
sorted_profit = economics.sort_values("Expected_Net_Profit", ascending=False).reset_index(drop=True)
cum_share = sorted_profit["Expected_Net_Profit"].cumsum() / sorted_profit["Expected_Net_Profit"].sum()
customer_share = (np.arange(len(sorted_profit)) + 1) / len(sorted_profit)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(customer_share, cum_share, color="#2b6cb0", linewidth=2, label="Actual concentration")
ax.plot([0, 1], [0, 1], color="#a0aec0", linestyle="--", label="Perfectly even distribution")
for mark in (0.25, 0.50):
    val = cum_share.iloc[int(mark * len(sorted_profit)) - 1]
    ax.scatter([mark], [val], color="#c53030", zorder=5)
    ax.annotate(f"Top {mark:.0%} of customers\n= {val:.0%} of profit",
                (mark, val), textcoords="offset points", xytext=(12, -22), fontsize=9)
ax.set_title("Concentration of expected profit across the 250 prospects")
ax.set_xlabel("Cumulative share of customers (ranked by expected profit)")
ax.set_ylabel("Cumulative share of expected profit")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

for mark in (0.10, 0.25, 0.50):
    idx = int(mark * len(sorted_profit)) - 1
    print(f"Top {mark:>4.0%} of customers ({idx+1:>3} names) account for "
          f"{cum_share.iloc[idx]:.1%} of expected profit")

### Business interpretation

Profit is **concentrated but not extreme**. The top quarter of the list
contributes just over half the expected profit, while the bottom quarter
contributes well under a tenth.

**What this means for budget decisions.** If the budget were halved, mailing the
top 125 names by expected profit would retain roughly three-quarters of the
expected profit at half the cost — raising ROI while reducing absolute return.
That is the trade-off management should be given explicitly, rather than a single
take-it-or-leave-it number.

## 6. Does every catalog pay for itself?

Before ranking customers, check whether any prospect destroys value. The
break-even test: a catalog pays for itself when

```
Predicted Sale Amount x P(response) x Gross Margin  >  $6.50
```

In [ ]:
mean_prob = economics["Response_Probability"].mean()
be_sale = pf.breakeven_sale_amount(mean_prob, assumptions.cost_per_catalog, assumptions.gross_margin)

unprofitable = (economics["Expected_Net_Profit"] <= 0).sum()
print(f"Break-even sale amount at the average response probability "
      f"({mean_prob:.1%}): ${be_sale:,.2f}")
print(f"Lowest predicted sale amount on the list:  ${economics['Predicted_Sale_Amount'].min():,.2f}")
print(f"Lowest expected net profit on the list:    ${economics['Expected_Net_Profit'].min():,.2f}")
print(f"\nProspects with non-positive expected net profit: {unprofitable} of {len(economics)}")

### Business interpretation

**Every one of the 250 prospects is individually profitable.** The least
attractive still returns $5.79 above the cost of their catalog, and the lowest
predicted sale amount ($125) is more than three times the $38 break-even
threshold.

This has a direct consequence for the recommendation: **there is no subset of
this list worth excluding**. Trimming the tail would reduce total profit. Targeting
only becomes relevant if the budget is capped — which is what the prioritisation
below is for.

## 7. Customer prioritisation

### How the thresholds are set

Arbitrary cut-offs ("top 50 customers", "anyone above $100") go stale the moment
the model or the assumptions change. The rule here derives from the profit
distribution itself (BRule-08):

| Tier | Rule | Business reasoning |
|---|---|---|
| **Low** | Expected net profit at or below zero | The catalog does not pay for itself. Mailing destroys value regardless of how the customer looks on other measures. |
| **High** | Profitable **and** at or above the 75th percentile of profitable customers | The top quartile by economic value. Mail first, and protect from budget cuts. |
| **Medium** | Profitable, below the 75th percentile | Worth mailing whenever budget allows. |

Because the threshold is a percentile of the current distribution rather than a
fixed dollar figure, it moves automatically when the model is refreshed or the
margin assumption changes.

In [ ]:
prioritised = pf.prioritise_customers(economics)
high_cut = prioritised["Priority_Threshold_High"].iloc[0]

priority = (prioritised.groupby("Priority")
            .agg(Customers=("Customer_ID", "count"),
                 Expected_Revenue=("Expected_Revenue", "sum"),
                 Expected_Net_Profit=("Expected_Net_Profit", "sum"),
                 Min_Profit=("Expected_Net_Profit", "min"),
                 Max_Profit=("Expected_Net_Profit", "max"))
            .reindex(["High", "Medium", "Low"]).dropna(how="all"))
priority["Campaign_Cost"] = priority["Customers"] * assumptions.cost_per_catalog
priority["ROI"] = priority["Expected_Net_Profit"] / priority["Campaign_Cost"]
priority["Share_of_profit"] = priority["Expected_Net_Profit"] / priority["Expected_Net_Profit"].sum()

print(f"High-priority threshold (75th percentile of profitable customers): ${high_cut:,.2f}\n")
priority

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colours = {"High": "#2f855a", "Medium": "#2b6cb0", "Low": "#c53030"}
for tier in ["High", "Medium", "Low"]:
    sub = prioritised[prioritised["Priority"] == tier]
    if len(sub):
        ax.scatter(sub["Predicted_Sale_Amount"], sub["Response_Probability"],
                   s=sub["Expected_Net_Profit"] * 0.8, alpha=0.55,
                   color=colours[tier], label=f"{tier} (n={len(sub)})")
ax.set_title("Prospect value map: predicted sale amount vs response probability\n(bubble size = expected net profit)")
ax.set_xlabel("Predicted sale amount ($)"); ax.set_ylabel("P(response)")
ax.legend(title="Priority")
plt.tight_layout(); plt.show()

### Business interpretation

**63 High-priority prospects (25% of the list) carry $11,746 — 53% of total
expected profit.** If the budget were cut to a quarter, mailing only these names
would retain over half the return for a quarter of the cost.

**No prospect falls into the Low tier**, because none has a non-positive expected
profit. The tier is retained in the framework rather than dropped: the rule must
still work when applied to a future list that does contain value-destroying names,
and a framework that only handles the convenient case is not a framework.

**The value map shows why ranking on either input alone would be wrong.** The
most valuable prospects are not simply those with the highest predicted spend, nor
those most likely to respond. They sit where both are reasonably high. A prospect
with a $2,000 predicted sale and an 18% response probability is worth less to the
campaign than one with an $800 sale and a 70% probability. Only the product of the
two identifies the right targets — which is precisely what a segment-intuition
approach would miss.

In [ ]:
top_20 = (prioritised.sort_values("Expected_Net_Profit", ascending=False)
          .head(20)[["Profit_Rank", "Customer_ID", "Customer_Segment",
                     "Predicted_Sale_Amount", "Response_Probability",
                     "Expected_Revenue", "Expected_Net_Profit", "Customer_ROI", "Priority"]])
top_20.reset_index(drop=True)

## 8. Export for reporting and BI

Flat CSV extracts for the dashboard, documented in
[`dashboard/dashboard_data_dictionary.md`](../dashboard/dashboard_data_dictionary.md).

In [ ]:
out_dir = Path.cwd().parent / "outputs"
out_dir.mkdir(exist_ok=True)

export_cols = ["Customer_ID", "Name", "Customer_Segment", "City", "State", "ZIP",
               "Store_Number", "Avg_Num_Products_Purchased", "Years_as_Customer",
               "Predicted_Sale_Amount", "Response_Probability", "Expected_Revenue",
               "Gross_Profit", "Catalog_Cost", "Expected_Net_Profit", "Customer_ROI",
               "Priority", "Profit_Rank"]

prioritised[export_cols].to_csv(out_dir / "customer_scores.csv", index=False)
pd.DataFrame([summary]).to_csv(out_dir / "campaign_summary.csv", index=False)
segment.to_csv(out_dir / "segment_summary.csv", index=False)
priority.reset_index().to_csv(out_dir / "priority_summary.csv", index=False)

print("Written to outputs/:")
for f in ["customer_scores.csv", "campaign_summary.csv", "segment_summary.csv", "priority_summary.csv"]:
    print(f"  - {f}")

## Summary

| Measure | Value |
|---|---|
| Customers mailed | 250 |
| Expected revenue | $47,224.87 |
| Gross profit (50% margin) | $23,612.44 |
| Campaign cost | $1,625.00 |
| **Expected net profit** | **$21,987.44** |
| Campaign ROI | 13.53x |
| Profitable prospects | 250 of 250 |
| High-priority prospects | 63 (53% of expected profit) |

All three source-project figures were independently reproduced from the supplied
data.

**These figures depend on assumptions that may not hold.** Notebook 04 tests how
far they can move before the recommendation changes:
[`04_sensitivity_analysis.ipynb`](04_sensitivity_analysis.ipynb).